In [ ]:
import torch
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# 定义相同的模型结构
class DeepSurvBRCA(torch.nn.Module):
    def __init__(self, input_dim=500):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_dim, 256)
        self.bn1 = torch.nn.BatchNorm1d(256)
        self.fc2 = torch.nn.Linear(256, 64)
        self.bn2 = torch.nn.BatchNorm1d(64)
        self.dropout = torch.nn.Dropout(0.4)
        self.output = torch.nn.Linear(64, 1)

    def forward(self, x):
        x = self.dropout(torch.nn.functional.relu(self.bn1(self.fc1(x))))
        x = self.dropout(torch.nn.functional.relu(self.bn2(self.fc2(x))))
        return self.output(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepSurvBRCA().to(device)
model.load_state_dict(torch.load('DeepSurv_BRCA_Best_0.7446.pth', map_location=device))
model.eval()

scaler = joblib.load('scaler.pkl')
top_500_genes = joblib.load('top_500_genes_list.pkl')
X_test = pd.read_csv('X_test_final.csv', index_col=0)
y_test = pd.read_csv('y_test_final.csv', index_col=0)

In [ ]:
with torch.no_grad():
    X_test_scaled = scaler.transform(X_test)
    risk_scores = model(torch.tensor(X_test_scaled, dtype=torch.float32).to(device)).cpu().numpy().flatten()

def plot_km(cutoff_percentile, title):
    threshold = np.percentile(risk_scores, cutoff_percentile)
    high_risk = risk_scores >= threshold
    
    kmf = KaplanMeierFitter()
    plt.figure(figsize=(8, 5))
    
    # 绘图逻辑
    kmf.fit(y_test['PFI.time'][high_risk], y_test['PFI'][high_risk], label=f'High Risk (Top {100-cutoff_percentile}%)')
    kmf.plot_survival_function(color='red', ci_show=True)
    kmf.fit(y_test['PFI.time'][~high_risk], y_test['PFI'][~high_risk], label='Lower Risk')
    kmf.plot_survival_function(color='blue', ci_show=True)
    
    res = logrank_test(y_test['PFI.time'][high_risk], y_test['PFI.time'][~high_risk],
                        y_test['PFI'][high_risk], y_test['PFI'][~high_risk])
    
    plt.title(f"{title}\nLog-rank P-value: {res.p_value:.2e}")
    plt.xlabel("Days")
    plt.ylabel("PFI Probability")
    plt.show()

plot_km(75, "Model Stratification: Top 25% vs Others")
plot_km(50, "Model Stratification: Median Split (50%)")

In [ ]:
def predict_patient_risk(new_patient_data):
    # 对齐特征
    data_500 = new_patient_data[top_500_genes]
    # 标准化 (不使用 .values 以保持警告清晰，或使用以避开警告)
    data_scaled = scaler.transform(data_500)
    
    with torch.no_grad():
        inputs = torch.tensor(data_scaled, dtype=torch.float32).to(device)
        scores = model(inputs).cpu().numpy().flatten()
    
    # 注意：这里的 Risk_Level 是基于测试集分布判定的
    # 在真实部署中，我们会预设一个固定的数值阈值
    results = pd.DataFrame({
        'Risk_Score': scores,
        'Risk_Level': ['High Risk' if s > np.median(risk_scores) else 'Low Risk' for s in scores]
    }, index=new_patient_data.index)
    return results

# 示例展示
print("Example Predictions:")
print(predict_patient_risk(X_test.head(5)))